# Notebook 05 — Inundación NASA (Global Flood Database + JRC + Sentinel-1) sobre AOI acotado

Caracteriza la dinámica de inundación dentro del AOI acotado (SFF CGSM + Vía Parque Isla de Salamanca, 835 km²) combinando tres fuentes: el Global Flood Database del Dartmouth Flood Observatory para eventos históricos, el JRC Global Surface Water Explorer para transiciones de superficie de agua y Sentinel-1 GRD para detección de inundación en el episodio La Niña de septiembre-octubre 2020. Esta es la **versión vigente**; el notebook `05_flooding_nasa.ipynb` original conserva el cálculo sobre AOI envolvente (5.073 km²) y se mantiene en la serie legacy para trazabilidad.

**Insumos:** `data/raw/cgsm_aoi_acotado_4326.geojson`
**Productos:** `outputs/tables/gfd_eventos_cgsm_acotado.csv`, `jrc_transiciones_cgsm_acotado.csv`, `inundacion_sar_sept2020_acotado.csv`, `eventos_inundacion_estaciones_acotado.csv`

In [1]:
from pathlib import Path
from datetime import datetime
import ee
import geopandas as gpd
import pandas as pd

try:
    ee.Initialize(project='basic-buttress-338101')
except Exception:
    import google.auth
    creds, _ = google.auth.default()
    ee.Initialize(credentials=creds, project='basic-buttress-338101')

ROOT = Path('..').resolve()
OUT_DIR = ROOT / 'outputs' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# AOI acotado oficial (RUNAP: SFF CGSM + VPI Isla de Salamanca)
gdf_aoi = gpd.read_file(ROOT / 'data' / 'raw' / 'cgsm_aoi_acotado_4326.geojson')
if gdf_aoi.crs is None or gdf_aoi.crs.to_epsg() != 4326:
    gdf_aoi = gdf_aoi.to_crs(4326)
geom_union = gdf_aoi.geometry.unary_union
aoi = ee.Geometry(geom_union.__geo_interface__)
print(f'AOI acotado: {len(gdf_aoi)} polígono(s), área aproximada {geom_union.area * 12321:.0f} km² (referencia esférica)')

AOI acotado: 1 polígono(s), área aproximada 852 km² (referencia esférica)


/tmp/ipykernel_3898/1069432397.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom_union = gdf_aoi.geometry.unary_union


## 1. Global Flood Database — eventos históricos sobre el AOI acotado

La colección `GLOBAL_FLOOD_DB/MODIS_EVENTS/V1` del Dartmouth Flood Observatory registra eventos de inundación entre 2000 y 2018 a partir de MODIS (250 m). Se calcula el área inundada de cada evento restringida al AOI acotado y se exporta la tabla.

In [2]:
gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1').filterBounds(aoi)
events_list = gfd.toList(50)
n_events = events_list.size().getInfo()
print(f'Eventos GFD que intersecan el AOI acotado: {n_events}')

flood_records = []
for i in range(n_events):
    img = ee.Image(events_list.get(i))
    idx = img.get('system:index').getInfo()
    parts = idx.split('_')
    start_date = f'{parts[3][:4]}-{parts[3][4:6]}-{parts[3][6:]}'
    end_date   = f'{parts[5][:4]}-{parts[5][4:6]}-{parts[5][6:]}'
    flood_band = img.select('flooded').clip(aoi)
    area = flood_band.eq(1).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=250, maxPixels=1e13
    ).get('flooded').getInfo()
    km2 = (area or 0) / 1e6
    duracion = (datetime.strptime(end_date, '%Y-%m-%d') - datetime.strptime(start_date, '%Y-%m-%d')).days
    flood_records.append({
        'evento': idx, 'inicio': start_date, 'fin': end_date,
        'area_inundada_km2': round(km2, 2), 'duracion_dias': duracion,
    })
    print(f'  {idx[:30]:<30} {start_date} → {end_date}  {km2:>7.2f} km²')

pd.DataFrame(flood_records).to_csv(OUT_DIR / 'gfd_eventos_cgsm_acotado.csv', index=False)
print('Guardado: outputs/tables/gfd_eventos_cgsm_acotado.csv')

Eventos GFD que intersecan el AOI acotado: 16
  DFO_1818_From_20011026_to_2001 2001-10-26 → 2001-11-15     0.00 km²
  DFO_1996_From_20020720_to_2002 2002-07-20 → 2002-07-31   233.94 km²
  DFO_2588_From_20041120_to_2004 2004-11-20 → 2004-11-27   274.82 km²
  DFO_2625_From_20050211_to_2005 2005-02-11 → 2005-02-26   299.20 km²
  DFO_2753_From_20051019_to_2005 2005-10-19 → 2005-10-25     0.00 km²
  DFO_2761_From_20050915_to_2005 2005-09-15 → 2005-12-16   239.40 km²
  DFO_2823_From_20060322_to_2006 2006-03-22 → 2006-06-02    16.51 km²
  DFO_3080_From_20070519_to_2007 2007-05-19 → 2007-06-27    15.76 km²
  DFO_3212_From_20071001_to_2007 2007-10-01 → 2007-12-10   211.47 km²
  DFO_3311_From_20080527_to_2008 2008-05-27 → 2008-05-28    78.76 km²
  DFO_3421_From_20081213_to_2008 2008-12-13 → 2008-12-14   151.89 km²
  DFO_3452_From_20090212_to_2009 2009-02-12 → 2009-02-16     5.25 km²
  DFO_3750_From_20101115_to_2010 2010-11-15 → 2010-12-20   190.06 km²
  DFO_3754_From_20101125_to_2010 2010-11-25 

## 2. JRC Global Surface Water — transiciones dentro del AOI acotado

Cada celda de 30 m de la banda `transition` codifica la historia hídrica entre 1984 y 2021. Se cuantifica el área de cada clase de transición y la estacionalidad media anual restringidas al AOI acotado.

In [3]:
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(aoi)
transition  = jrc.select('transition')
occurrence  = jrc.select('occurrence')
seasonality = jrc.select('seasonality')

trans_classes = {
    1: 'Permanente', 2: 'Nuevo permanente', 3: 'Permanente perdido',
    4: 'Estacional', 5: 'Nuevo estacional', 6: 'Estacional perdido',
    7: 'Estacional a permanente', 8: 'Permanente a estacional',
    9: 'Efimero permanente', 10: 'Efimero estacional',
}
trans_records = []
for code, name in trans_classes.items():
    area = transition.eq(code).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
    ).get('transition').getInfo()
    km2 = (area or 0) / 1e6
    if km2 > 0:
        trans_records.append({'clase': name, 'codigo': code, 'area_km2': round(km2, 2)})
        print(f'  {name:<30} {km2:>8.2f} km²')

pd.DataFrame(trans_records).to_csv(OUT_DIR / 'jrc_transiciones_cgsm_acotado.csv', index=False)

perm = occurrence.gt(75).multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).get('occurrence').getInfo() / 1e6
seas = occurrence.gt(25).And(occurrence.lte(75)).multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).get('occurrence').getInfo() / 1e6
print(f'\nAgua permanente (>75% del tiempo): {perm:.2f} km²')
print(f'Agua estacional (25-75%):          {seas:.2f} km²')
print('Guardado: outputs/tables/jrc_transiciones_cgsm_acotado.csv')

  Permanente                       377.75 km²
  Nuevo permanente                  23.55 km²
  Permanente perdido                18.55 km²
  Estacional                        17.98 km²
  Nuevo estacional                  31.31 km²
  Estacional perdido                29.38 km²
  Estacional a permanente           10.93 km²
  Permanente a estacional           18.54 km²
  Efimero permanente                 1.74 km²
  Efimero estacional                49.30 km²

Agua permanente (>75% del tiempo): 385.34 km²
Agua estacional (25-75%):          44.12 km²
Guardado: outputs/tables/jrc_transiciones_cgsm_acotado.csv


## 3. Sentinel-1 SAR — inundación bajo dosel en septiembre-octubre 2020

Se comparan medianas trimestrales de la banda VH del Sentinel-1 GRD entre el período seco (enero-marzo 2020) y el episodio La Niña (septiembre-octubre 2020). En zonas de manglar, la inundación bajo dosel produce un aumento del backscatter VH por el efecto de doble rebote, de modo que una diferencia negativa (seco − inundado < −2 dB) sugiere agua bajo la cobertura forestal. Para agua abierta, el comportamiento es el opuesto.

In [4]:
s1_dry = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi).filterDate('2020-01-01', '2020-03-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select('VH').median().clip(aoi))
s1_flood = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi).filterDate('2020-09-01', '2020-10-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select('VH').median().clip(aoi))
sar_diff = s1_dry.subtract(s1_flood).rename('SAR_diff')

flood_mask  = sar_diff.gt(3).selfMask().rename('flood')
underst_msk = sar_diff.lt(-2).selfMask().rename('understory')

flood_area = flood_mask.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=10, maxPixels=1e13
).get('flood').getInfo() / 1e6
underst_area = underst_msk.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=10, maxPixels=1e13
).get('understory').getInfo() / 1e6

print(f'Agua abierta (SAR diff > +3 dB):     {flood_area:.2f} km²')
print(f'Inundación bajo dosel (< −2 dB):     {underst_area:.2f} km²')
print(f'Total afectado:                      {(flood_area + underst_area):.2f} km²')

pd.DataFrame({
    'tipo': ['Agua abierta (SAR diff > 3 dB)',
             'Bajo dosel manglar (SAR diff < -2 dB)', 'Total'],
    'area_km2': [round(flood_area, 2), round(underst_area, 2),
                 round(flood_area + underst_area, 2)],
}).to_csv(OUT_DIR / 'inundacion_sar_sept2020_acotado.csv', index=False)
print('Guardado: outputs/tables/inundacion_sar_sept2020_acotado.csv')

Agua abierta (SAR diff > +3 dB):     15.93 km²
Inundación bajo dosel (< −2 dB):     43.08 km²
Total afectado:                      59.02 km²
Guardado: outputs/tables/inundacion_sar_sept2020_acotado.csv


## 4. Cruce con estaciones de monitoreo

Para cada estación se reporta el SAR diff medio en un buffer de 500 m y se cuentan los eventos históricos del GFD que la intersecan.

In [7]:
stations = {
    'Isla Boqueron':  (-74.298, 10.962),
    'Punta Cerro':    (-74.283, 10.973),
    'Punta Chino':    (-74.305, 10.912),
    'Rio Sevilla':    (-74.325, 10.880),
    'Cano Palos':     (-74.471, 10.758),
    'CP Pajarales':   (-74.75,  10.85),
    'Cano Clarin':    (-74.55,  10.55),
    'VIPIS':          (-74.65,  11.02),
}
rows = []
for name, (lon, lat) in stations.items():
    pt = ee.Geometry.Point([lon, lat]).buffer(500)
    diff_val = sar_diff.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=pt, scale=10, maxPixels=1e8
    ).get('SAR_diff').getInfo()
    n_inundado = 0
    max_ev = ''
    for i in range(n_events):
        img = ee.Image(events_list.get(i))
        idx = img.get('system:index').getInfo()
        v = img.select('flooded').reduceRegion(
            reducer=ee.Reducer.max(), geometry=pt, scale=250, maxPixels=1e8
        ).get('flooded').getInfo()
        if v and v > 0:
            n_inundado += 1
            if not max_ev: max_ev = idx[:25]

    rows.append({
        'estacion': name,
        'sar_diff_db': round(diff_val, 2) if diff_val is not None else None,
        'dentro_aoi_acotado': diff_val is not None,
        'eventos_gfd_2001_2017': n_inundado,
        'evento_referencia': max_ev,
    })

    sar_str = f'{diff_val:>+6.2f} dB' if diff_val is not None else '   N/A     '
    print(f'  {name:<14} sar_diff={sar_str}   {n_inundado} eventos GFD')

pd.DataFrame(rows).to_csv(OUT_DIR / 'eventos_inundacion_estaciones_acotado.csv', index=False)
print('\nGuardado: outputs/tables/eventos_inundacion_estaciones_acotado.csv')

  Isla Boqueron  sar_diff=   N/A        10 eventos GFD
  Punta Cerro    sar_diff=   N/A        10 eventos GFD
  Punta Chino    sar_diff=   N/A        9 eventos GFD
  Rio Sevilla    sar_diff=   N/A        10 eventos GFD
  Cano Palos     sar_diff= +0.15 dB   2 eventos GFD
  CP Pajarales   sar_diff=   N/A        9 eventos GFD
  Cano Clarin    sar_diff=   N/A        1 eventos GFD
  VIPIS          sar_diff= +0.12 dB   9 eventos GFD

Guardado: outputs/tables/eventos_inundacion_estaciones_acotado.csv
